# RoadScan: reproducible detector training

Updated 11 October 2026. The first T4 run completed 37 epochs, with low Czech validation performance (mAP50 0.0612). Its temporary checkpoint was lost after the runtime reset; the saved log is retained separately. The recovery notebook exports a best-checkpoint ZIP every five epochs and after final validation. Download and hash-check exports promptly; do not rely on temporary storage or notebook output persistence alone.

1. Select a T4 GPU runtime. Upload **RoadScan-Colab-Dataset-Bootstrap.zip** using the left Files pane.
2. Run setup, reconstruction and verification cells in order. The bootstrap fetches official public archives and requires every selected image/label to match the recorded local provenance. The earlier large browser ZIP transfer failed validation and was not used.
3. Run training once; existing runs are protected. Download checkpoint links as they appear. Final validation and final export run automatically in the training cell.

No Drive mount, RoadScan access key, private photograph or paid API key is needed. Training: India/China MotorBike; validation: Czech. US final-country images remain separate and must not be used for model/threshold tuning. Saudi final testing requires separately supplied images. Synthetic software tests do not measure detector accuracy.

Classes: D00 longitudinal crack, D10 transverse crack, D20 alligator crack, D40 pothole. RDD2022 by Arya, Maeda, Ghosh, Toshniwal and Sekimoto: https://doi.org/10.6084/m9.figshare.21431547.v1 (CC BY 4.0). Ultralytics: AGPL-3.0; https://www.ultralytics.com/license . Training API: https://docs.ultralytics.com/modes/train/ .


In [ ]:
import sys, subprocess, json, hashlib, platform
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Select a Colab GPU runtime before continuing; refusing a long CPU run."
subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics==8.4.175"])
import ultralytics
from ultralytics import YOLO, settings
settings.update({key: False for key in ("sync", "wandb", "mlflow", "clearml", "comet", "neptune", "hub", "tensorboard") if key in settings})
print({"python":platform.python_version(), "torch":torch.__version__, "ultralytics":ultralytics.__version__, "gpu":torch.cuda.get_device_name(0)})


In [ ]:
import json, hashlib, zipfile, stat, sys, subprocess
from pathlib import Path
BOOTSTRAP = Path('/content/RoadScan-Colab-Dataset-Bootstrap.zip')
BOOTSTRAP_SHA = '7a706c0869c546f1c94e1e626fcd79285bae1ca41c3ae4e5d46197b2e5966af5'
assert hashlib.sha256(BOOTSTRAP.read_bytes()).hexdigest() == BOOTSTRAP_SHA, 'Bootstrap transfer checksum failed'
BOOT_ROOT = Path('/content/roadscan_bootstrap')
assert not BOOT_ROOT.exists(), 'Bootstrap exists; inspect prior run before retrying'
allowed = {'remote_rdd_index.py','download_rdd_countries.py','rebuild_dataset.py','expected-sources.json','expected-provenance.json','work/model-start/archive-index.json','roadscan/__init__.py','roadscan/rdd_prepare.py','roadscan/detector_train.py'}
with zipfile.ZipFile(BOOTSTRAP) as archive:
    assert archive.testzip() is None
    assert set(archive.namelist()) == allowed and len(archive.infolist()) == len(allowed)
    assert all(not stat.S_ISLNK(item.external_attr >> 16) and item.file_size < 32*1024*1024 for item in archive.infolist())
    archive.extractall(BOOT_ROOT)
subprocess.check_call([sys.executable,'rebuild_dataset.py'],cwd=BOOT_ROOT)
BASE=Path('/content/roadscan_training')
provenance=json.loads((BASE/'provenance.json').read_text())
TRAIN_ROOT=BASE/'training'
data_yaml=TRAIN_ROOT/'data.yaml'
EXPECTED_SHA256='98b6bfa8e9e5b28112bb42654c2c6c688e42f5e50c9d024c27c30711aae6878e'
DATA_TRANSFER_METHOD='verified public source archive reconstruction; every selected image/label matched local provenance'
print('Verified reconstruction completed.')


In [ ]:
assert provenance['class_order'] == ['D00','D10','D20','D40']
assert len(provenance['records']) == 9000
for record in provenance['records']:
    assert record['split'] in ('train','val'), 'Held-out data must not enter training'
    for kind in ('image','label'):
        item=BASE/record[kind]
        assert item.resolve().is_relative_to(BASE.resolve())
        assert item.is_file() and hashlib.sha256(item.read_bytes()).hexdigest() == record[kind+'_sha256']
print(provenance['country_plan'])
print(provenance['class_box_counts'])
print('Training/validation provenance verified; held-out images excluded.')


In [ ]:
TOOLS=Path("/content/roadscan_tools/roadscan")
TOOLS.mkdir(parents=True,exist_ok=True)
(TOOLS/"__init__.py").write_text("")
(TOOLS/"detector_train.py").write_text('"""Validate a local YOLO dataset and optionally train an offline detector.\n\nDirectory layout: root/{train,val}/{images,labels}/stem.{jpg,png,...|txt}.\nTest images must never appear in this training root. Training requires a\nlocally supplied YOLO model checkpoint and an optional Ultralytics install.\n"""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nCLASSES = (\'D00\', \'D10\', \'D20\', \'D40\')\nEXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.webp\'}\n\n\ndef inspect_dataset(root):\n    root = Path(root).resolve(strict=True)\n    if not root.is_dir():\n        raise ValueError(\'Dataset root must be a directory\')\n    if (root / \'test\').exists():\n        raise ValueError(\'Keep final test images outside the training dataset root\')\n    counts = {}\n    seen_hashes = {}\n    for split in (\'train\', \'val\'):\n        images_dir = root / split / \'images\'\n        labels_dir = root / split / \'labels\'\n        if not images_dir.is_dir() or not labels_dir.is_dir():\n            raise ValueError(f\'Missing {split}/images or {split}/labels\')\n        images = sorted(p for p in images_dir.iterdir() if p.suffix.lower() in EXTENSIONS)\n        if not images:\n            raise ValueError(f\'No images in {split}\')\n        stems = set()\n        for image in images:\n            if image.is_symlink() or not image.is_file() or image.stem in stems:\n                raise ValueError(f\'Unsafe or ambiguous image: {image}\')\n            stems.add(image.stem)\n            label = labels_dir / (image.stem + \'.txt\')\n            if label.is_symlink() or not label.is_file():\n                raise ValueError(f\'Missing label file: {label}\')\n            digest = hashlib.sha256(image.read_bytes()).hexdigest()\n            if digest in seen_hashes and seen_hashes[digest] != split:\n                raise ValueError(\'Exact duplicate image across train and val\')\n            seen_hashes[digest] = split\n            for line_number, line in enumerate(label.read_text(encoding=\'utf-8\').splitlines(), 1):\n                if not line.strip():\n                    continue\n                tokens = line.split()\n                if len(tokens) != 5:\n                    raise ValueError(f\'Invalid YOLO annotation: {label}:{line_number}\')\n                try:\n                    class_id = int(tokens[0])\n                    coords = [float(value) for value in tokens[1:]]\n                except ValueError as exc:\n                    raise ValueError(f\'Invalid YOLO annotation: {label}:{line_number}\') from exc\n                import math\n                if not 0 <= class_id < 4 or not all(math.isfinite(x) for x in coords):\n                    raise ValueError(f\'Invalid class or nonfinite coordinate: {label}:{line_number}\')\n                cx, cy, w, h = coords\n                # Decimal serialization can put a boundary a few floating-point ULPs outside 0/1.\n                tolerance = 1e-9\n                if not (0 < w <= 1 and 0 < h <= 1 and -tolerance <= cx-w/2 and cx+w/2 <= 1+tolerance and -tolerance <= cy-h/2 and cy+h/2 <= 1+tolerance):\n                    raise ValueError(f\'Out-of-bounds box: {label}:{line_number}\')\n        extra_labels = {p.stem for p in labels_dir.glob(\'*.txt\')} - stems\n        if extra_labels:\n            raise ValueError(f\'Orphan labels in {split}: {sorted(extra_labels)[0]}\')\n        counts[split] = len(images)\n    return {\'root\': str(root), \'classes\': list(CLASSES), \'image_counts\': counts,\n            \'warning\': \'Exact hashes only; separately audit sequence, country, and near-duplicate leakage.\'}\n\n\ndef run_training(root, weights, epochs, imgsz, project, name):\n    if epochs < 1 or imgsz < 32:\n        raise ValueError(\'epochs must be >= 1 and imgsz >= 32\')\n    info = inspect_dataset(root)\n    checkpoint = Path(weights).resolve(strict=True)\n    if not checkpoint.is_file() or checkpoint.suffix.lower() != \'.pt\':\n        raise ValueError(\'Supply a local .pt checkpoint; no automatic downloads\')\n    destination = Path(project).resolve()\n    if (destination / name).exists():\n        raise ValueError(\'Refusing to overwrite an existing experiment directory\')\n    try:\n        from ultralytics import YOLO\n    except ImportError as exc:\n        raise RuntimeError(\'Install optional ultralytics after reviewing its license\') from exc\n    # Use an absolute dataset YAML written inside the experiment directory.\n    destination.mkdir(parents=True, exist_ok=True)\n    experiment = destination / name\n    experiment.mkdir(exist_ok=False)\n    yaml_path = experiment / \'dataset.yaml\'\n    yaml_path.write_text(\'path: \' + json.dumps(info[\'root\']) + \'\\ntrain: train/images\\nval: val/images\\nnames:\\n\' +\n                         \'\'.join(f\'  {i}: {cls}\\n\' for i, cls in enumerate(CLASSES)), encoding=\'utf-8\')\n    (experiment / \'dataset_audit.json\').write_text(json.dumps(info, indent=2) + \'\\n\', encoding=\'utf-8\')\n    model = YOLO(str(checkpoint))\n    model.train(data=str(yaml_path), epochs=epochs, imgsz=imgsz, device=\'cpu\',\n                project=str(destination), name=name, exist_ok=True)\n    return info\n\n\ndef main(argv=None):\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--dataset-root\', required=True)\n    parser.add_argument(\'--weights\', help=\'Existing local YOLO .pt checkpoint; required for --train\')\n    parser.add_argument(\'--train\', action=\'store_true\', help=\'Run optional CPU training after audit\')\n    parser.add_argument(\'--epochs\', type=int, default=30)\n    parser.add_argument(\'--imgsz\', type=int, default=640)\n    parser.add_argument(\'--project\', default=\'experiments/detector_runs\')\n    parser.add_argument(\'--name\', default=\'e2_local\')\n    args = parser.parse_args(argv)\n    try:\n        if args.train:\n            if not args.weights:\n                parser.error(\'--train requires --weights\')\n            result = run_training(args.dataset_root, args.weights, args.epochs, args.imgsz, args.project, args.name)\n        else:\n            result = inspect_dataset(args.dataset_root)\n        print(json.dumps(result, indent=2))\n    except (ValueError, OSError, RuntimeError) as exc:\n        parser.error(str(exc))\n\n\nif __name__ == \'__main__\':\n    main()\n')
sys.path.insert(0,str(TOOLS.parent))
from roadscan.detector_train import inspect_dataset
audit=inspect_dataset(TRAIN_ROOT)
assert audit["image_counts"]=={"train":8000,"val":1000}
print(audit)


In [ ]:
import os, io, zipfile, base64, shutil
from IPython.display import HTML, display, update_display
from ultralytics.utils.torch_utils import strip_optimizer
RUNS = Path('/content/roadscan_runs')
RUN_NAME = 'E2_recovery_yolo11n_640_seed20261010'
assert not (RUNS / RUN_NAME).exists(), 'Experiment exists; preserve it before another run.'
os.chdir('/content')
model = YOLO('yolo11n.pt')
initial_checkpoint = Path(model.ckpt_path)
train_args = {'data': str(data_yaml), 'epochs': 50, 'imgsz': 640, 'batch': 16,
    'device': 0, 'workers': 2, 'seed': 20261010, 'deterministic': True,
    'project': str(RUNS), 'name': RUN_NAME, 'exist_ok': False, 'patience': 10,
    'cache': False, 'plots': True, 'optimizer': 'AdamW', 'lr0': 0.001,
    'degrees': 0.0, 'translate': 0.05, 'scale': 0.25, 'fliplr': 0.5, 'flipud': 0.0,
    'mosaic': 0.5, 'mixup': 0.0, 'close_mosaic': 10}
manifest = {'initial_checkpoint': initial_checkpoint.name,
    'initial_checkpoint_sha256': hashlib.sha256(initial_checkpoint.read_bytes()).hexdigest(),
    'local_reference_training_package_sha256': EXPECTED_SHA256,
    'dataset_transfer_method': DATA_TRANSFER_METHOD, 'bootstrap_sha256': BOOTSTRAP_SHA,
    'all_training_files_matched_local_provenance': True, 'train_args': train_args,
    'python': platform.python_version(), 'torch': torch.__version__,
    'ultralytics': ultralytics.__version__, 'gpu': torch.cuda.get_device_name(0),
    'source': provenance['source'], 'final_test_used': False, 'status': 'training_started',
    'purpose': 'Reproduce initial experiment after temporary Colab runtime lost checkpoint; no test-set tuning'}
RUNS.mkdir(exist_ok=True)
(RUNS/(RUN_NAME+'_manifest.json')).write_text(json.dumps(manifest, indent=2))
export_state = {'shown': False}
def export_best(epoch, final=False):
    run = RUNS/RUN_NAME
    best_path = run/'weights/best.pt'
    if not best_path.is_file():
        return
    copy_path = Path('/content/roadscan_export_best.pt')
    shutil.copy2(best_path, copy_path)
    strip_optimizer(copy_path)
    manifest['exported_after_epoch'] = int(epoch)
    manifest['best_checkpoint_sha256'] = hashlib.sha256(copy_path.read_bytes()).hexdigest()
    manifest['status'] = 'trained_and_validation_measured' if final else 'intermediate_training_checkpoint'
    buffer = io.BytesIO()
    with zipfile.ZipFile(buffer, 'w', zipfile.ZIP_DEFLATED) as archive:
        archive.write(copy_path, 'best.pt')
        archive.writestr('experiment_manifest.json', json.dumps(manifest, indent=2))
        archive.writestr('dataset_provenance.json', json.dumps(provenance))
        for filename in ('results.csv', 'args.yaml', 'validation_metrics.json'):
            path = run/filename
            if path.exists():
                archive.write(path, filename)
    name = 'RoadScan-Trained-Detector-Artifacts.zip' if final else f'RoadScan-Checkpoint-Epoch-{epoch}.zip'
    payload = buffer.getvalue()
    Path('/content', name).write_bytes(payload)
    digest = hashlib.sha256(payload).hexdigest()
    markup = HTML(f'<p>Epoch {epoch}; final={final}; bytes={len(payload)}; SHA256={digest}</p><a download="{name}" href="data:application/zip;base64,{base64.b64encode(payload).decode()}">Download {name}</a>')
    if export_state['shown']:
        update_display(markup, display_id='roadscan-checkpoint-export')
    else:
        display(markup, display_id='roadscan-checkpoint-export')
        export_state['shown'] = True
    print(f'EXPORTED {name} SHA256 {digest}', flush=True)
def checkpoint_callback(trainer):
    epoch = trainer.epoch + 1
    if epoch % 5 == 0 or trainer.stop:
        export_best(epoch)
model.add_callback('on_model_save', checkpoint_callback)
model.train(**train_args)
BEST = RUNS/RUN_NAME/'weights/best.pt'
best = YOLO(str(BEST))
assert [best.names[i] for i in range(4)] == ['D00','D10','D20','D40'] and len(best.names) == 4
metrics = best.val(data=str(data_yaml), split='val', device=0, imgsz=640, plots=True,
    project=str(RUNS), name=RUN_NAME+'_validation', exist_ok=False)
per_class = {best.names[int(class_id)]: {'AP50': float(metrics.box.ap50[position]),
    'AP50_95': float(metrics.box.ap[position].mean())}
    for position, class_id in enumerate(metrics.box.ap_class_index.tolist())}
measured = {'cohort': 'Czech validation; used for model selection, not final test',
    'mAP50': float(metrics.box.map50), 'mAP50_95': float(metrics.box.map),
    'mean_precision': float(metrics.box.mp), 'mean_recall': float(metrics.box.mr),
    'per_class': per_class, 'validation_device': 'Colab T4 GPU',
    'speed_ms_per_image': metrics.speed, 'laptop_cpu_latency_measured': False,
    'heldout_country_evaluated': False, 'local_saudi_test_evaluated': False}
(RUNS/RUN_NAME/'validation_metrics.json').write_text(json.dumps(measured, indent=2))
print(json.dumps(measured, indent=2))
export_best(model.trainer.epoch + 1, final=True)


In [ ]:
# Read-only check: training cell already performed final validation/export.
assert (RUNS/RUN_NAME/'validation_metrics.json').is_file(), 'Final validation has not completed.'
print((RUNS/RUN_NAME/'validation_metrics.json').read_text())
artifact=Path('/content/RoadScan-Trained-Detector-Artifacts.zip')
assert artifact.is_file(), 'Final artifact export missing.'
print('Final artifact SHA256:',hashlib.sha256(artifact.read_bytes()).hexdigest())


In [ ]:
# Optional full experiment archive including plots and environment versions.
import shutil, base64
from IPython.display import HTML, display
ARTIFACTS=Path('/content/RoadScan-Full-Training-Run.zip')
assert not ARTIFACTS.exists(), 'Archive already exists; preserve existing results.'
freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True)
(RUNS/RUN_NAME/'runtime_requirements.txt').write_text(freeze)
shutil.make_archive(str(ARTIFACTS.with_suffix('')),'zip',RUNS)
print('Artifact bytes:',ARTIFACTS.stat().st_size)
print('Artifact SHA256:',hashlib.sha256(ARTIFACTS.read_bytes()).hexdigest())
payload=base64.b64encode(ARTIFACTS.read_bytes()).decode('ascii')
display(HTML(f'<a download="{ARTIFACTS.name}" href="data:application/zip;base64,{payload}">Download full RoadScan training run</a>'))


## Next stage, after the artifact ZIP is saved locally

Codex will verify the checkpoint class mapping and its recorded hash, inspect validation error examples, measure CPU latency on this laptop, and integrate a compatible model into a separate test deployment before replacing the live model. Freeze the selected model/confidence threshold before evaluating the reserved United States and local Saudi test sets. Do not call validation scores final-test accuracy.

Experiments still required by the supplied project guide: size/resolution and augmentation comparisons, a genuine transfer-learning multilabel baseline, domain-shift evaluation, duplicate retrieval evaluation, and any requirements confirmed by the actual instructor rubric. The rubric itself was not provided in the handoff.
